# Q4 — Structured Streaming: **master** notebook

Live 1-minute edit counts per wiki with a 2-minute watermark, the Kafka offset/checkpoint
mechanism, and a restart that neither loses nor duplicates a row.

| phase | answers |
|---|---|
| 1 | `readStream` + `from_json` + event-time window + watermark → **memory sink** — **Q4a** |
| 2 | the same windows through `foreachBatch` → Parquet + dashboard topic — **Q4a** |
| 3 | `lastProgress` at a 10 s and a 30 s trigger → latency/throughput trade-off — **Q4b** |
| 4 | restart on the same checkpoint: offsets continue, no replay, no gap — **Q4c** |
| 5 | a deliberate *fresh* checkpoint run, which does replay — the contrast that proves the point |

**Before you start:** the queries read from `latest`, so they only see *live* traffic.
Run the producer in a JupyterLab terminal for the whole notebook:

```
python3 q1_wikimedia_producer.py --minutes 8
```

**Why the query runs in `local[2]`.** A checkpoint is the offset log *plus* the aggregation
state, and both are written by the executors — they must land on a filesystem the driver can
read back. Every volume in this stack is per node, so a driver in `skc-jupyter` with executors
on other machines would scatter state across machines and the Q4c restart would be
unreliable. One JVM with 2 cores (the same as one worker) keeps the checkpoint consistent;
set `Q4_MASTER=spark://10.8.0.6:7077` if your workers share storage (HDFS/S3). The
distributed parts of the pipeline are Q1 (ingest) and Q2/Q3 (batch).

Everything is written under `/tmp/spark-events/q4/` on that one node. **Each full run starts
from a clean work directory** (the session cell wipes it): a memory sink cannot recover from
a checkpoint, and the dashboard query would otherwise continue from yesterday's offsets
instead of demonstrating Q4c again. Set `Q4_KEEP_WORK=1` before starting the kernel if you
want to reuse the previous checkpoint and sink.

> The demo writes to `wiki.recentchange.raw`. Running it grows that topic, so if you also
> want Q2/Q3 numbers to keep matching, run those first — or re-run `q2_spark_master.ipynb`
> afterwards to refresh `master_results.json`.


In [7]:
import json, os, re, shutil, time, pathlib, urllib.request

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               IntegerType, BooleanType)

# ------------------------------------------------------------------ config
BOOTSTRAP  = os.environ.get("KAFKA_BOOTSTRAP", "10.8.0.6:29092")
TOPIC      = os.environ.get("Q4_TOPIC", "wiki.recentchange.raw")
DASH_TOPIC = os.environ.get("Q4_DASH_TOPIC", "wiki.edit.windows")

WORK       = pathlib.Path(os.environ.get("Q4_WORK", "/tmp/spark-events/q4"))
CKPT       = WORK / "checkpoints"
SINK       = WORK / "windows"              # parquet, one folder per (re)start append
SINK_NOCK  = WORK / "windows_no_checkpoint"  # the deliberate contrast run
RESULT     = WORK / "q4_results.json"

WINDOW_S    = "1 minute"
WATERMARK_S = "2 minutes"

# A Structured Streaming checkpoint (offset log + aggregation state) and a local file sink
# must live on a filesystem the driver AND every executor can read. This lab's volumes are
# per node (README: "Volumes are per node"), so driver and executors stay in one JVM for the
# checkpoint to stay consistent. local[2] = 2 cores, the same as one worker. Point
# Q4_MASTER at spark://10.8.0.6:7077 if your workers share storage (HDFS/S3).
STREAM_MASTER = os.environ.get("Q4_MASTER", "local[2]")

# how long each phase runs (seconds) — kept in one place so a fast test can shorten them
RUN_MEM_SECONDS     = 200
RUN_10S_SECONDS     = 90
RUN_30S_SECONDS     = 150
RUN_CONTRAST_SECONDS = 40

# ------------------------------------------------------------------ schema (same as Q3)
EVENT_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("type", StringType()),
    StructField("wiki", StringType()),
    StructField("server_name", StringType()),
    StructField("title", StringType()),
    StructField("namespace", IntegerType()),
    StructField("user", StringType()),
    StructField("bot", BooleanType()),
    StructField("minor", BooleanType()),
    StructField("timestamp", LongType()),
    StructField("comment", StringType()),
    StructField("length", StructType([StructField("old", LongType()),
                                      StructField("new", LongType())])),
])

# ------------------------------------------------------------------ helpers
def table(rows, cols, title="", max_rows=None):
    def get(r, c):
        if hasattr(r, "asDict"):
            r = r.asDict(recursive=True)
        if isinstance(r, dict):
            return r.get(c, "")
        try:
            return r[c]
        except Exception:                        # noqa: BLE001
            return ""

    rows = list(rows)
    shown = rows if max_rows is None else rows[:max_rows]
    w = {c: max(len(str(c)), 6, *(len(str(get(r, c))) for r in shown)) if shown else len(str(c))
         for c in cols}
    out = [title] if title else []
    out.append("  ".join(f"{str(c):>{w[c]}}" for c in cols))
    out.append("-" * (sum(w.values()) + 2 * (len(cols) - 1)))
    for r in shown:
        out.append("  ".join(f"{str(get(r, c)):>{w[c]}}" for c in cols))
    if max_rows is not None and len(rows) > max_rows:
        out.append(f"... {len(rows) - max_rows} more rows")
    text = chr(10).join(out)
    print(text)
    return text


def progress_row(p):
    dur = p.get("durationMs") or {}
    et = p.get("eventTime") or {}
    return {
        "batch": p.get("batchId"),
        "at": (p.get("timestamp") or "")[11:19],
        "input rows": p.get("numInputRows"),
        "input rows/s": round(p.get("inputRowsPerSecond") or 0, 1),
        "processed rows/s": round(p.get("processedRowsPerSecond") or 0, 1),
        "trigger ms": dur.get("triggerExecution", 0),
        "addBatch ms": dur.get("addBatch", 0),
        "watermark": (et.get("watermark") or "")[:19],
        "late rows dropped": sum(s.get("numRowsDroppedByWatermark", 0)
                                 for s in (p.get("stateOperators") or [])),
    }


def progress_table(query, title="lastProgress per batch"):
    rows = [progress_row(p) for p in (query.recentProgress or [])]
    table(rows, ["batch", "at", "input rows", "input rows/s", "processed rows/s",
                 "trigger ms", "batch watermark", "late rows dropped"], title=title)
    return rows


def run_for(query, seconds, label, trigger="n/a"):
    """Keep a background streaming query alive for `seconds`, printing live progress."""
    print(f"[{label}] running {seconds}s — trigger {trigger}")
    t0 = time.time()
    while time.time() - t0 < seconds:
        time.sleep(max(0.5, min(10, seconds - (time.time() - t0))))
        p = query.lastProgress
        if not p:
            print(f"  t+{time.time() - t0:5.0f}s  no batch yet")
            continue
        dur = (p.get("durationMs") or {}).get("triggerExecution", 0)
        print(f"  t+{time.time() - t0:5.0f}s  batch {p['batchId']:>3}: "
              f"input={p['numInputRows']:>6} rows ({p['inputRowsPerSecond']:>7.1f}/s)  "
              f"processed={p['processedRowsPerSecond']:>7.1f}/s  trigger={dur:>5} ms")
    print(f"[{label}] stopping")
    query.stop()
    print(f"[{label}] stopped — isActive={query.isActive}")


def checkpoint_batches(ckpt_dir):
    """Checkpoint offsets/N files: batch watermark + Kafka end offsets per committed batch."""
    off = pathlib.Path(ckpt_dir) / "offsets"
    rows = []
    if not off.is_dir():
        return rows
    for f in sorted(off.iterdir(), key=lambda p: int(p.name) if p.name.isdigit() else -1):
        if not f.name.isdigit():
            continue
        lines = [ln for ln in f.read_text().splitlines() if ln.strip()]
        meta = offsets = None
        for ln in lines[1:]:
            try:
                obj = json.loads(ln)
            except Exception:                    # noqa: BLE001
                continue
            if isinstance(obj, dict) and "batchWatermarkMs" in obj:
                meta = obj
            elif isinstance(obj, dict):
                offsets = obj
        wm = "none"
        if meta and meta.get("batchWatermarkMs"):
            wm = time.strftime("%H:%M:%S", time.localtime(meta["batchWatermarkMs"] / 1000.0))
        topic_map = next(iter(offsets.values()), {}) if offsets else {}
        rows.append({"batch": int(f.name), "batch watermark": wm,
                     "end offsets": ", ".join(f"P{k}={v}" for k, v in sorted(
                         topic_map.items(), key=lambda kv: int(kv[0])))})
    return rows


def window_batch_writer(sink_dir, dash_topic, batch_log):
    """foreachBatch body: append this micro-batch to Parquet and to the dashboard topic."""
    def writer(batch_df, batch_id):
        n = batch_df.count()
        record = {"batch": batch_id, "rows": n,
                  "at": time.strftime("%H:%M:%S"), "kafka": "skipped"}
        if n:
            (batch_df.write.mode("append").parquet(str(sink_dir)))
            try:
                (batch_df.selectExpr("wiki AS key", "to_json(struct(*)) AS value")
                        .write.format("kafka")
                        .option("kafka.bootstrap.servers", BOOTSTRAP)
                        .option("topic", dash_topic)
                        .save())
                record["kafka"] = f"{n} rows -> {dash_topic}"
            except Exception as exc:             # noqa: BLE001
                record["kafka"] = f"failed: {type(exc).__name__}: {exc}"
        batch_log.append(record)
        print(f"    foreachBatch {batch_id}: {n} window rows  ({record['kafka']})")
    return writer


In [8]:
spark = (SparkSession.builder
         .master(STREAM_MASTER)
         .appName("q4-structured-streaming")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

# Re-running the notebook must not inherit the previous run's checkpoints: a *memory* sink
# cannot recover from one ("This query does not support recovering from checkpoint
# location"), and the foreachBatch query would silently continue from the old offsets.
# The whole work dir is therefore reset here, once per run. Set Q4_KEEP_WORK=1 to keep the
# previous sink + checkpoints and continue instead of starting over.
if os.environ.get("Q4_KEEP_WORK") != "1":
    shutil.rmtree(WORK, ignore_errors=True)
    print("work dir reset:", WORK, "(set Q4_KEEP_WORK=1 to reuse it)")
WORK.mkdir(parents=True, exist_ok=True)
print("spark version :", spark.version)
print("sc.master     :", sc.master, "(driver+executors in one JVM: checkpoint-safe)")
print("parallelism   :", sc.defaultParallelism)
print("application id:", sc.applicationId)
print("driver UI     :", sc.uiWebUrl)
print("topic         :", TOPIC, "-> dashboard topic:", DASH_TOPIC)
print("window        :", WINDOW_S, "| watermark:", WATERMARK_S)
print("checkpoints   :", CKPT)
print("parquet sink  :", SINK)

work dir reset: /tmp/spark-events/q4 (set Q4_KEEP_WORK=1 to reuse it)
spark version : 4.1.3
sc.master     : local[2] (driver+executors in one JVM: checkpoint-safe)
parallelism   : 4
application id: local-1794842553242
driver UI     : http://10.8.0.6:4041
topic         : wiki.recentchange.raw -> dashboard topic: wiki.edit.windows
window        : 1 minute | watermark: 2 minutes
checkpoints   : /tmp/spark-events/q4/checkpoints
parquet sink  : /tmp/spark-events/q4/windows


## Q4a — the stream: `readStream` → `from_json` → event-time windows

* `startingOffsets=latest`: the query starts at the live end of the topic. On a **restart**
  the checkpoint wins and this option is ignored — that is the behaviour Q4c depends on.
* `event_ts` is derived from the payload's own `timestamp` (Unix seconds), **not** from the
  Kafka record timestamp: event time is what the producer observed at Wikimedia, so late or
  re-ordered delivery does not change which window a record belongs to.
* 1-minute tumbling windows, 2-minute watermark: Spark keeps state for `max(event time) −
  2 min`, so a window is finalised and emitted only after the watermark passes its end.

The aggregation is `edits`, `bot_edits` and the derived `bot_pct` per (window, wiki).


In [9]:
raw = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", BOOTSTRAP)
       .option("subscribe", TOPIC)
       .option("startingOffsets", "latest")
       .option("maxOffsetsPerTrigger", 20000)
       .load())

events = (raw
          .select(F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"),
                  F.col("partition").alias("kafka_partition"),
                  F.col("offset").alias("kafka_offset"))
          .select("e.*", "kafka_partition", "kafka_offset")
          .withColumn("event_ts", F.col("timestamp").cast("timestamp"))  # epoch s -> timestamp
          .filter(F.col("type") == "edit")
          .filter(F.col("wiki").isNotNull()))

windows = (events
           .withWatermark("event_ts", WATERMARK_S)
           .groupBy(F.window(F.col("event_ts"), WINDOW_S).alias("win"), "wiki")
           .agg(F.count("*").alias("edits"),
                F.sum(F.when(F.col("bot"), 1).otherwise(0)).alias("bot_edits"))
           .select(F.col("win.start").alias("window_start"),
                   F.col("win.end").alias("window_end"),
                   "wiki",
                   "edits",
                   "bot_edits",
                   F.round(100.0 * F.col("bot_edits") / F.col("edits"), 2).alias("bot_pct")))

print("schema of the streaming aggregation:")
windows.printSchema()
print("isStreaming:", windows.isStreaming)

schema of the streaming aggregation:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- wiki: string (nullable = true)
 |-- edits: long (nullable = false)
 |-- bot_edits: long (nullable = true)
 |-- bot_pct: double (nullable = true)

isStreaming: True


### Phase 1 — memory sink (this is the "console or memory" output)

`format("memory")` + `queryName` makes the result queryable as a temp table while the query
runs — the interactive equivalent of the console sink. `outputMode("append")` on a windowed
aggregation requires the watermark, which is exactly why the watermark is not optional here.


In [10]:
RUN_IDS, QID = {}, {}
# The memory sink is a view, not durable output: it cannot recover from a checkpoint, so a
# leftover one makes Spark refuse to start ("... does not support recovering from checkpoint
# location"). Wiping it is safe — the aggregation is rebuilt from the live stream.
shutil.rmtree(CKPT / "mem", ignore_errors=True)
(CKPT / "mem").mkdir(parents=True, exist_ok=True)
q_mem = (windows.writeStream
         .format("memory")
         .queryName("wiki_windows")
         .outputMode("append")
         .option("checkpointLocation", str(CKPT / "mem"))
         .trigger(processingTime="10 seconds")
         .start())

QID["memory"] = q_mem.id
RUN_IDS["memory"] = q_mem.runId
print("memory-sink query id:", q_mem.id, "| runId:", q_mem.runId)
run_for(q_mem, RUN_MEM_SECONDS, "memory sink", "10 s")
print()
print("the table is in the catalog as 'wiki_windows'; it is read in Phase 4, once the")
print("watermark has passed some window ends (a 1-minute window needs ~3 minutes of stream)")


26/11/16 15:27:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/16 15:27:00 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


memory-sink query id: 70014ea0-65f4-47a5-b29e-403144c8b25a | runId: e7587620-ba67-4c11-805b-101daa6533c7
[memory sink] running 200s — trigger 10 s
  t+   10s  batch   0: input=     0 rows (    0.0/s)  processed=    0.0/s  trigger=  604 ms
  t+   20s  batch   1: input=   237 rows (   24.2/s)  processed=  138.4/s  trigger= 1713 ms
  t+   30s  batch   2: input=   222 rows (   22.2/s)  processed=  164.9/s  trigger= 1346 ms
  t+   40s  batch   3: input=   368 rows (   36.8/s)  processed=  228.7/s  trigger= 1609 ms
  t+   50s  batch   4: input=   373 rows (   37.3/s)  processed=  195.4/s  trigger= 1908 ms
  t+   60s  batch   5: input=   273 rows (   27.3/s)  processed=  194.9/s  trigger= 1401 ms
  t+   70s  batch   6: input=   285 rows (   28.5/s)  processed=  222.3/s  trigger= 1281 ms
  t+   80s  batch   7: input=   336 rows (   33.6/s)  processed=  175.0/s  trigger= 1920 ms
  t+   90s  batch   8: input=   479 rows (   47.9/s)  processed=  288.6/s  trigger= 1659 ms
  t+  100s  batch   9: in

26/11/16 15:30:20 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 20, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@c899db0] is aborting.
26/11/16 15:30:20 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 20, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@c899db0] aborted.
26/11/16 15:30:20 WARN DAGScheduler: Failed to cancel job group e7587620-ba67-4c11-805b-101daa6533c7. Cannot find active jobs for it.
26/11/16 15:30:20 WARN TaskSetManager: Lost task 2.0 in stage 43.0 (TID 291) (10.8.0.6 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 21 cancelled Query wiki_windows [id = 70014ea0-65f4-47a5-b29e-403144c8b25a, runId = e7587620-ba67-4c11-805b-101daa6533c7] was stopped SQLSTATE: XXKDA)
26/11/16 15:30:20 WARN TaskSetManager: Lost task 3.0 in stage 43.0 (TID 292) (10.8.0.6 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED]

### Phase 2 — `foreachBatch` for the dashboard (Parquet + Kafka)

`foreachBatch` hands each micro-batch to ordinary batch code, which is what makes a
dashboard possible without a special connector: the same rows are appended to Parquet (for
the audit in Q4c) and published to `wiki.edit.windows` as `key = wiki`, `value = JSON`, so
any consumer can render it. The checkpoint for this query lives in `checkpoints/dash` and is
**reused** by every restart below — that is the object Q4c is about.


In [11]:
BATCH_LOG = []
(CKPT / "dash").mkdir(parents=True, exist_ok=True)

q_dash = (windows.writeStream
          .outputMode("append")
          .foreachBatch(window_batch_writer(SINK, DASH_TOPIC, BATCH_LOG))
          .option("checkpointLocation", str(CKPT / "dash"))
          .trigger(processingTime="10 seconds")
          .start())

QID["first"], RUN_IDS["dash_10s"] = q_dash.id, q_dash.runId
print("query id:", q_dash.id, "| runId:", q_dash.runId)
run_for(q_dash, RUN_10S_SECONDS, "dashboard sink", "10 s")

prog_10 = progress_table(q_dash, "Q4b — progress at a 10 s trigger")
table(BATCH_LOG, ["batch", "rows", "at", "kafka"],
      title="foreachBatch calls (what each micro-batch wrote)")

26/11/16 15:30:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/16 15:30:40 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


query id: b68c7298-41e0-43b8-80a2-6a1a17ee4a81 | runId: 4c5290e7-883e-4b69-88ce-a7009ff78a54
[dashboard sink] running 90s — trigger 10 s
    foreachBatch 0: 0 window rows  (skipped)
  t+   10s  batch   0: input=     0 rows (    0.0/s)  processed=    0.0/s  trigger=  378 ms
    foreachBatch 1: 0 window rows  (skipped)
  t+   20s  batch   1: input=   425 rows (   44.2/s)  processed=  315.5/s  trigger= 1346 ms
    foreachBatch 2: 0 window rows  (skipped)
  t+   30s  batch   2: input=   410 rows (   41.0/s)  processed=  401.2/s  trigger= 1022 ms
    foreachBatch 3: 0 window rows  (skipped)
  t+   40s  batch   3: input=   389 rows (   38.9/s)  processed=  258.0/s  trigger= 1508 ms
    foreachBatch 4: 0 window rows  (skipped)
  t+   50s  batch   4: input=   400 rows (   40.0/s)  processed=  292.6/s  trigger= 1367 ms
    foreachBatch 5: 0 window rows  (skipped)
  t+   60s  batch   5: input=   377 rows (   37.7/s)  processed=  274.4/s  trigger= 1374 ms
    foreachBatch 6: 0 window rows  (skipp

Exception in thread "stream execution thread for [id = b68c7298-41e0-43b8-80a2-6a1a17ee4a81, runId = 4c5290e7-883e-4b69-88ce-a7009ff78a54]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4761)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4816)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4925)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4725)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:3958)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4761)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4816)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:4925)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:4847)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4725)
	at java.base/java.util.regex.Pattern$C

'foreachBatch calls (what each micro-batch wrote)\n batch    rows        at    kafka\n---------------------------------\n     0       0  15:30:40  skipped\n     1       0  15:30:51  skipped\n     2       0  15:31:01  skipped\n     3       0  15:31:11  skipped\n     4       0  15:31:21  skipped\n     5       0  15:31:31  skipped\n     6       0  15:31:41  skipped\n     7       0  15:31:51  skipped\n     8       0  15:32:01  skipped'

26/11/16 15:32:10 WARN TaskSetManager: Lost task 3.0 in stage 72.0 (TID 425) (10.8.0.6 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 31 cancelled Query [id = b68c7298-41e0-43b8-80a2-6a1a17ee4a81, runId = 4c5290e7-883e-4b69-88ce-a7009ff78a54] was stopped SQLSTATE: XXKDA)
26/11/16 15:32:10 WARN TaskSetManager: Lost task 2.0 in stage 72.0 (TID 424) (10.8.0.6 executor driver): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 31 cancelled Query [id = b68c7298-41e0-43b8-80a2-6a1a17ee4a81, runId = 4c5290e7-883e-4b69-88ce-a7009ff78a54] was stopped SQLSTATE: XXKDA)


## Q4b — `lastProgress` at 10 s vs 30 s triggers

The same query is restarted with `trigger(processingTime="30 seconds")` on the **same
checkpoint**, so this is also the restart that Q4c verifies: the offset chain continues
instead of replaying.

Read the two tables for:

* `input rows/s` — arrival rate the source delivered in that batch;
* `processed rows/s` — what the query managed to process, i.e. throughput;
* `trigger ms` — wall time of the whole batch (planning + state store + sink writes);
* `input rows` — the batch size, which is why the two intervals behave differently.


In [12]:
q_dash_30 = (windows.writeStream
             .outputMode("append")
             .foreachBatch(window_batch_writer(SINK, DASH_TOPIC, BATCH_LOG))
             .option("checkpointLocation", str(CKPT / "dash"))   # SAME checkpoint
             .trigger(processingTime="30 seconds")
             .start())

QID["restart"], RUN_IDS["dash_30s"] = q_dash_30.id, q_dash_30.runId
print("query id after restart:", q_dash_30.id,
      "| unchanged:", q_dash_30.id == QID["first"])
print("runId after restart   :", q_dash_30.runId, "(per execution, expected to differ)")
run_for(q_dash_30, RUN_30S_SECONDS, "dashboard sink", "30 s")
prog_30 = progress_table(q_dash_30, "Q4b — progress at a 30 s trigger")


def stats(rows, key):
    vals = [r[key] for r in rows if r[key]]
    return (sum(vals) / len(vals)) if vals else 0.0


table([
    {"trigger": "10 s", "batches": len(prog_10),
     "rows/batch": round(stats(prog_10, "input rows"), 1),
     "input rows/s": round(stats(prog_10, "input rows/s"), 1),
     "processed rows/s": round(stats(prog_10, "processed rows/s"), 1),
     "trigger ms": round(stats(prog_10, "trigger ms"), 1)},
    {"trigger": "30 s", "batches": len(prog_30),
     "rows/batch": round(stats(prog_30, "input rows"), 1),
     "input rows/s": round(stats(prog_30, "input rows/s"), 1),
     "processed rows/s": round(stats(prog_30, "processed rows/s"), 1),
     "trigger ms": round(stats(prog_30, "trigger ms"), 1)},
], ["trigger", "batches", "rows/batch", "input rows/s", "processed rows/s", "trigger ms"],
    title="Q4b — the same query, two trigger intervals")
print()
print("How to read it: a trigger interval is the *minimum* latency between an event arriving")
print("and the dashboard seeing it. At 10 s the query wakes three times as often, so a batch")
print("carries ~1/3 of the rows, the fixed per-batch cost (query planning, state-store read/")
print("write, Parquet + Kafka sink setup) is paid three times as often, and end-to-end")
print("latency stays near 10 s + batch duration. At 30 s each batch is bigger, the fixed cost")
print("is amortised and rows/second goes up, at the price of up to 30 s of extra latency.")
print("So: shorter interval = lower latency and lower efficiency; longer interval = higher")
print("throughput and lazy dashboards. For a 1-minute window with a 2-minute watermark,")
print("10-15 s is the sweet spot: it keeps the window near-final while never letting the")
print("backlog grow past one or two batches.")

26/11/16 15:32:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/16 15:32:43 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


query id after restart: b68c7298-41e0-43b8-80a2-6a1a17ee4a81 | unchanged: True
runId after restart   : 0146061d-33ff-4cb0-8770-090dc84f28de (per execution, expected to differ)
[dashboard sink] running 150s — trigger 30 s


26/11/16 15:32:44 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=1, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 9 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:32:44 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=0, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 9 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:32:44 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/spark-events/q4/checkpoints/dash/state/0/1/9.snapshot, hence no checksum verification.
26/11/16 15:32:44 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/spa

    foreachBatch 9: 0 window rows  (skipped)
  t+   10s  batch   9: input=   360 rows (    0.0/s)  processed=  257.7/s  trigger= 1396 ms
    foreachBatch 10: 0 window rows  (skipped)
  t+   20s  batch  10: input=  1884 rows (  111.1/s)  processed= 1060.2/s  trigger= 1776 ms
  t+   30s  batch  10: input=  1884 rows (  111.1/s)  processed= 1060.2/s  trigger= 1776 ms
  t+   40s  batch  10: input=  1884 rows (  111.1/s)  processed= 1060.2/s  trigger= 1776 ms
    foreachBatch 11: 0 window rows  (skipped)
  t+   50s  batch  11: input=  1028 rows (   34.3/s)  processed=  959.9/s  trigger= 1071 ms
  t+   60s  batch  11: input=  1028 rows (   34.3/s)  processed=  959.9/s  trigger= 1071 ms
  t+   70s  batch  11: input=  1028 rows (   34.3/s)  processed=  959.9/s  trigger= 1071 ms


26/11/16 15:34:02 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=1, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 12 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:34:02 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=0, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 12 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:34:02 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/spark-events/q4/checkpoints/dash/state/0/1/12.snapshot, hence no checksum verification.
26/11/16 15:34:02 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/

    foreachBatch 12: 44 window rows  (44 rows -> wiki.edit.windows)
  t+   80s  batch  12: input=   971 rows (   32.4/s)  processed=  400.4/s  trigger= 2425 ms
  t+   90s  batch  12: input=   971 rows (   32.4/s)  processed=  400.4/s  trigger= 2425 ms
  t+  100s  batch  12: input=   971 rows (   32.4/s)  processed=  400.4/s  trigger= 2425 ms
    foreachBatch 13: 0 window rows  (skipped)
  t+  110s  batch  13: input=   840 rows (   28.0/s)  processed=  457.0/s  trigger= 1837 ms
  t+  120s  batch  13: input=   840 rows (   28.0/s)  processed=  457.0/s  trigger= 1837 ms
  t+  130s  batch  13: input=   840 rows (   28.0/s)  processed=  457.0/s  trigger= 1837 ms


26/11/16 15:35:00 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=1, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 14 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:35:00 WARN HDFSBackedStateStoreProvider StateStoreProviderId[ storeId=StateStoreId[ operatorId=0, partitionId=0, storeName=default ], queryRunId=0146061d-33ff-4cb0-8770-090dc84f28de ]: The state for version 14 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
26/11/16 15:35:00 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/spark-events/q4/checkpoints/dash/state/0/0/14.snapshot, hence no checksum verification.
26/11/16 15:35:00 WARN ChecksumCheckpointFileManager: No checksum file found for file:/tmp/

    foreachBatch 14: 63 window rows  (63 rows -> wiki.edit.windows)
  t+  140s  batch  14: input=     0 rows (    0.0/s)  processed=    0.0/s  trigger=  711 ms
  t+  150s  batch  14: input=     0 rows (    0.0/s)  processed=    0.0/s  trigger=  711 ms
[dashboard sink] stopping
[dashboard sink] stopped — isActive=False
Q4b — progress at a 30 s trigger
 batch        at  input rows  input rows/s  processed rows/s  trigger ms  batch watermark  late rows dropped
------------------------------------------------------------------------------------------------------------
     9  15:32:43         360             0             257.7        1396                                   0
    10  15:33:00        1884         111.1            1060.2        1776                                   0
    11  15:33:30        1028          34.3             959.9        1071                                   0
    12  15:34:00         971          32.4             400.4        2425                              

26/11/16 15:35:13 WARN DAGScheduler: Failed to cancel job group 0146061d-33ff-4cb0-8770-090dc84f28de. Cannot find active jobs for it.
26/11/16 15:35:13 WARN DAGScheduler: Failed to cancel job group 0146061d-33ff-4cb0-8770-090dc84f28de. Cannot find active jobs for it.


## Phase 4 — the memory-sink table, now that windows have closed

The Phase-1 query has been running for several minutes, so its watermark has passed some
window ends and those windows are in the table. This is the interactive counterpart of the
console sink: query it like any other temp view.


In [13]:
try:
    rows = spark.sql("""
        SELECT window_start, window_end, wiki, edits, bot_edits, bot_pct
        FROM wiki_windows ORDER BY window_start DESC, edits DESC
    """).collect()
    table(rows, ["window_start", "window_end", "wiki", "edits", "bot_edits", "bot_pct"],
          title=f"memory sink: wiki_windows ({len(rows)} rows, newest windows first)",
          max_rows=15)
except Exception as exc:                         # noqa: BLE001
    rows = []
    print("memory table not readable:", type(exc).__name__, exc)

print()
prog_mem = progress_table(q_mem, "Phase 1 query progress")
q_mem.stop()
print("Phase 1 query stopped — isActive:", q_mem.isActive)

memory sink: wiki_windows (10 rows, newest windows first)
       window_start           window_end          wiki   edits  bot_edits  bot_pct
----------------------------------------------------------------------------------
2026-11-16 15:26:00  2026-11-16 15:27:00        enwiki       7          0      0.0
2026-11-16 15:26:00  2026-11-16 15:27:00   commonswiki       5          2     40.0
2026-11-16 15:26:00  2026-11-16 15:27:00  wikidatawiki       3          0      0.0
2026-11-16 15:26:00  2026-11-16 15:27:00        arwiki       2          2    100.0
2026-11-16 15:26:00  2026-11-16 15:27:00        frwiki       2          0      0.0
2026-11-16 15:26:00  2026-11-16 15:27:00        jawiki       1          0      0.0
2026-11-16 15:26:00  2026-11-16 15:27:00  dewiktionary       1          0      0.0
2026-11-16 15:26:00  2026-11-16 15:27:00        ukwiki       1          1    100.0
2026-11-16 15:26:00  2026-11-16 15:27:00  jawiktionary       1          1    100.0
2026-11-16 15:25:00  2026-11-

## Q4c — restart on the same checkpoint: nothing lost, nothing duplicated

Four independent pieces of evidence:

1. **`runId` unchanged** — it is stored in the checkpoint, so a restart is the same logical
   query, not a new one.
2. **Offset chain** — the checkpoint's `offsets/N` files keep counting from the batch where
   the first run stopped; the Kafka end offsets per partition only move forward.
3. **Sink is not re-written** — every (window, wiki) row appears exactly once in the Parquet
   sink, although two runs wrote into it.
4. **Values are correct** — for every window the sink emitted, the count matches an
   independent batch recomputation over the same event-time range.


In [14]:
print("query id — first run   :", QID.get("first"))
print("query id — after restart:", QID.get("restart"))
print("same logical query     :", QID.get("first") == QID.get("restart"),
      "(the checkpoint's metadata file stores this id, so a restart continues the query)")
print("runId (per execution)  :", RUN_IDS.get("dash_10s"), "->", RUN_IDS.get("dash_30s"))
print()
table(checkpoint_batches(CKPT / "dash"), ["batch", "batch watermark", "end offsets"],
      title="checkpoint offsets/ — committed batch, watermark and Kafka end offsets")
print()
print("batch ids handed to foreachBatch across both runs:",
      [b["batch"] for b in BATCH_LOG])

# the offset chain must be unbroken: per partition the end offsets only ever move forward
chain = checkpoint_batches(CKPT / "dash")
parsed = {r["batch"]: {int(p.split("=")[0][1:]): int(p.split("=")[1])
                       for p in r["end offsets"].split(", ") if "=" in p} for r in chain}
batches = sorted(parsed)
monotonic = all(all(parsed[b][p] >= parsed[batches[i]][p] for p in parsed[b])
                for i, b in enumerate(batches) if i)
print("checkpoint batches:", batches,
      "| consecutive:", batches == list(range(batches[0], batches[0] + len(batches))),
      "| end offsets monotonic per partition:", monotonic)

query id — first run   : b68c7298-41e0-43b8-80a2-6a1a17ee4a81
query id — after restart: b68c7298-41e0-43b8-80a2-6a1a17ee4a81
same logical query     : True (the checkpoint's metadata file stores this id, so a restart continues the query)
runId (per execution)  : 4c5290e7-883e-4b69-88ce-a7009ff78a54 -> 0146061d-33ff-4cb0-8770-090dc84f28de

checkpoint offsets/ — committed batch, watermark and Kafka end offsets
 batch  batch watermark                                             end offsets
-------------------------------------------------------------------------------
     0             none  P0=4131, P1=3765, P2=6139, P3=30920, P4=4204, P5=52385
     1             none  P0=4147, P1=3772, P2=6190, P3=31122, P4=4215, P5=52523
     2         15:28:49  P0=4159, P1=3786, P2=6215, P3=31297, P4=4235, P5=52687
     3         15:28:59  P0=4175, P1=3797, P2=6231, P3=31471, P4=4251, P5=52843
     4         15:29:09  P0=4185, P1=3806, P2=6254, P3=31666, P4=4279, P5=52978
     5         15:29:19  P0=4

### The cross-check needs a static snapshot

Comparing the streamed windows against an independent computation means reading the same
topic **once, in batch**. This is the Q3 path (`from_json` with the explicit schema), now
just a temp view — it is the reference the streaming result is checked against, and it is
also why the two engines cannot silently disagree about what "one edit" means.


In [15]:
batch_raw = (spark.read.format("kafka")
             .option("kafka.bootstrap.servers", BOOTSTRAP)
             .option("subscribe", TOPIC)
             .option("startingOffsets", "earliest")
             .option("endingOffsets", "latest")
             .load())

(batch_raw
 .select(F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"))
 .select("e.*")
 .withColumn("event_ts", F.col("timestamp").cast("timestamp"))
 .filter(F.col("wiki").isNotNull())
 .createOrReplaceTempView("wiki_edits_batch"))

print("batch rows for the cross-check:",
      spark.sql("SELECT COUNT(*) AS n FROM wiki_edits_batch").first()["n"])
print("edit rows:",
      spark.sql("SELECT COUNT(*) AS n FROM wiki_edits_batch WHERE type='edit'").first()["n"])

batch rows for the cross-check: 109868
edit rows: 54874


In [16]:
sink = spark.read.parquet(str(SINK)).cache()   # one listing, rest off cache
n_sink = sink.count()
sink.createOrReplaceTempView("sink_rows")
print("window rows in the sink (both runs):", n_sink)

n_dupes = spark.sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT window_start, wiki, COUNT(*) AS copies
        FROM sink_rows GROUP BY window_start, wiki HAVING COUNT(*) > 1
    )
""").first()["n"]
print("duplicated (window, wiki) keys     :", n_dupes,
      "-> no window was emitted twice" if n_dupes == 0 else "-> DUPLICATES!")

totals = sink.agg(F.min("window_start").alias("lo"),
                  F.max("window_end").alias("hi"),
                  F.sum("edits").alias("edits")).first()
print(f"event-time range covered           : {totals['lo']} -> {totals['hi']}")
print(f"edits counted by the stream        : {totals['edits']:,}")

lo, hi = totals["lo"], totals["hi"]
drops = sum(p.get("late rows dropped", 0) for p in prog_mem + prog_10 + prog_30)
print(f"late rows dropped by the watermark : {drops}")

# A query that starts at `latest` never saw events produced before it started, so any
# window overlapping its start is partial by construction. The cut is the wall clock of the
# first micro-batch (batch 0) of the dashboard query.
q0_at = prog_10[0]["at"] if prog_10 and prog_10[0].get("at") else None
cut = f"{lo.strftime('%Y-%m-%d')} {q0_at}" if q0_at else lo.strftime('%Y-%m-%d %H:%M:%S')
print(f"values compared from               : {cut} (batch 0 of the dashboard query)")

static = spark.sql(f"""
    SELECT window(event_ts, '{WINDOW_S}').start AS window_start,
           wiki,
           COUNT(*)                             AS edits_static
    FROM wiki_edits_batch
    WHERE type = 'edit'
      AND event_ts >= TIMESTAMP'{lo.strftime('%Y-%m-%d %H:%M:%S')}'
      AND event_ts <  TIMESTAMP'{hi.strftime('%Y-%m-%d %H:%M:%S')}'
    GROUP BY window(event_ts, '{WINDOW_S}'), wiki
""")
static.createOrReplaceTempView("static_windows")
# both sides are then restricted to windows that started after the query did
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW static_cut AS
    SELECT * FROM static_windows WHERE window_start >= TIMESTAMP'{cut}'
""")

cmp = spark.sql(f"""
    SELECT COUNT(*)                                                            AS compared,
           SUM(CASE WHEN s.edits = t.edits_static THEN 1 ELSE 0 END)            AS matching,
           SUM(CASE WHEN s.edits IS DISTINCT FROM t.edits_static THEN 1 ELSE 0 END) AS mismatching,
           SUM(CASE WHEN t.edits_static IS NULL THEN 1 ELSE 0 END)              AS only_in_stream,
           SUM(CASE WHEN s.edits IS NULL THEN 1 ELSE 0 END)                     AS only_in_batch
    FROM (SELECT * FROM sink_rows WHERE window_start >= TIMESTAMP'{cut}') s
    FULL OUTER JOIN static_cut t
      ON s.window_start = t.window_start AND s.wiki = t.wiki
""").first()
table([cmp.asDict()], ["compared", "matching", "mismatching", "only_in_stream", "only_in_batch"],
      title="Q4c — stream vs independent batch recomputation (same event-time range)")

mism = spark.sql(f"""
    SELECT COALESCE(s.window_start, t.window_start) AS window_start,
           COALESCE(s.wiki, t.wiki)                 AS wiki,
           s.edits                                  AS stream_edits,
           t.edits_static
    FROM (SELECT * FROM sink_rows WHERE window_start >= TIMESTAMP'{cut}') s
    FULL OUTER JOIN static_cut t
      ON s.window_start = t.window_start AND s.wiki = t.wiki
    WHERE s.edits IS DISTINCT FROM t.edits_static
    ORDER BY window_start, wiki
""").collect()
if mism:
    table(mism, ["window_start", "wiki", "stream_edits", "edits_static"],
          title=f"rows that differ ({len(mism)}), first 10", max_rows=10)
print()
print(f"windows compared: {cmp['compared']}, exact matches: {cmp['matching']}, "
      f"differences: {cmp['mismatching']}")
print("windows that ended inside the last watermark are not in the sink yet — that is the")
print("watermark holding state, not a gap: the same query with a shorter watermark emits")
print("them earlier and with a longer one later.")

window rows in the sink (both runs): 107
duplicated (window, wiki) keys     : 0 -> no window was emitted twice
event-time range covered           : 2026-11-16 15:30:00 -> 2026-11-16 15:32:00
edits counted by the stream        : 2,081
late rows dropped by the watermark : 0
values compared from               : 2026-11-16 15:30:40 (batch 0 of the dashboard query)
Q4c — stream vs independent batch recomputation (same event-time range)
compared  matching  mismatching  only_in_stream  only_in_batch
--------------------------------------------------------------
      63        63            0               0              0

windows compared: 63, exact matches: 63, differences: 0
windows that ended inside the last watermark are not in the sink yet — that is the
watermark holding state, not a gap: the same query with a shorter watermark emits
them earlier and with a longer one later.


### The contrast: a run with a **fresh** checkpoint

Same query, same topic, a new empty checkpoint directory and `startingOffsets=earliest`.
Nothing remembers where the previous run stopped, so the query starts again at the
**beginning** of the topic — the offset log and the aggregation state *are* the checkpoint.
`trigger(availableNow=True)` lets that first batch finish on its own instead of waiting for
the clock; the two checkpoints' first-batch offsets are then compared side by side.


In [17]:
# The contrast is about where a run *starts*, and that is recorded in the checkpoint.
# A fresh checkpoint + startingOffsets=earliest has nothing to resume from, so it reads the
# topic from its first available offset again; the reused checkpoint resumes at the offsets
# it had already committed. trigger(availableNow) makes that first batch terminate on its
# own instead of waiting for the clock.
q_nock = (windows.writeStream
          .outputMode("append")
          .foreachBatch(window_batch_writer(SINK_NOCK, DASH_TOPIC, BATCH_LOG))
          .option("checkpointLocation", str(CKPT / "nockpt"))       # fresh checkpoint
          .option("startingOffsets", "earliest")                    # replay the history
          .trigger(availableNow=True)
          .start())
print("[contrast] first batch with a fresh checkpoint (availableNow)…")
q_nock.awaitTermination(timeout=RUN_CONTRAST_SECONDS * 8)
if q_nock.isActive:
    q_nock.stop()

resumed = checkpoint_batches(CKPT / "dash")
fresh = checkpoint_batches(CKPT / "nockpt")
table([{"checkpoint": "reused (dash)", "first batch": resumed[0]["batch"],
        "first batch end offsets": resumed[0]["end offsets"]},
       {"checkpoint": "fresh (nockpt)", "first batch": fresh[0]["batch"] if fresh else "-",
        "first batch end offsets": fresh[0]["end offsets"] if fresh else "-"}],
      ["checkpoint", "first batch", "first batch end offsets"],
      title="where the two runs start (the checkpoint decides, not startingOffsets)")
def _offset_sum(row):
    return sum(int(p.split("=")[1]) for p in row["end offsets"].split(", ") if "=" in p)


if fresh:
    print()
    print(f"checkpointed run: batch 0 ended at offsets summing to {_offset_sum(resumed[0]):,} "
          f"-> it continued from where the previous run stopped")
    print(f"fresh checkpoint: batch 0 ended at offsets summing to {_offset_sum(fresh[0]):,} "
          f"-> it read the topic from its earliest offsets, a full replay")

both = 0
if SINK_NOCK.exists():
    spark.read.parquet(str(SINK_NOCK)).createOrReplaceTempView("nock_rows")
    both = spark.sql("""
        SELECT COUNT(*) AS n FROM (
            SELECT window_start, wiki, COUNT(*) AS copies
            FROM (SELECT window_start, wiki FROM sink_rows
                  UNION ALL SELECT window_start, wiki FROM nock_rows)
            GROUP BY window_start, wiki HAVING COUNT(*) > 1
        )
    """).first()["n"]
print("windows present in BOTH sinks (i.e. replayed):", both)

[contrast] first batch with a fresh checkpoint (availableNow)…


26/11/16 15:35:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/16 15:35:33 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


    foreachBatch 0: 0 window rows  (skipped)
where the two runs start (the checkpoint decides, not startingOffsets)
    checkpoint  first batch                                 first batch end offsets
-----------------------------------------------------------------------------------
 reused (dash)            0  P0=4131, P1=3765, P2=6139, P3=30920, P4=4204, P5=52385
fresh (nockpt)            0  P0=4429, P1=4002, P2=6575, P3=34406, P4=4555, P5=55901

checkpointed run: batch 0 ended at offsets summing to 101,544 -> it continued from where the previous run stopped
fresh checkpoint: batch 0 ended at offsets summing to 109,868 -> it read the topic from its earliest offsets, a full replay
windows present in BOTH sinks (i.e. replayed): 0


## What the three mechanisms each do

* **Kafka offsets** are the source's bookkeeping: "this query has consumed partition P up to
  offset N". They are what makes the position *explicit* rather than implied by a socket.
* **`checkpointLocation`** is Spark's durable log of exactly that — the offsets of the last
  committed batch, the aggregation state (open windows and their counts), and the metadata
  (including `runId`). On restart Spark reads the log, resumes from the committed offsets and
  rebuilds the state, so each record is processed once and each closed window is emitted
  once. It also survives a crash *between* processing and commit: the batch is simply
  re-run, which is at-least-once at the sink and exactly-once per window because the state
  and the offsets are committed together.
* **The watermark** (`withWatermark("event_ts", "2 minutes")`) bounds the state: a window is
  emitted once the watermark passes its end, and records that arrive later than
  `max(event time) − 2 min` are dropped rather than opening an already-closed window again.
  That is the latency/accuracy dial: a larger watermark tolerates more late data and holds
  more state; a smaller one reacts faster and discards more stragglers. The
  `late rows dropped` column in the progress tables is that counter.

Late data inside the window still counts — it is aggregated into the open window; late data
beyond the watermark is dropped and reported. Both are visible in `lastProgress`, which is
why the column is in the tables above.


In [18]:
out = {
    "app_id": sc.applicationId,
    "topic": TOPIC,
    "dashboard_topic": DASH_TOPIC,
    "window": WINDOW_S, "watermark": WATERMARK_S,
    "query_id": {"first": QID.get("first"), "restart": QID.get("restart")},
    "query_id_stable": QID.get("first") == QID.get("restart"),
    "run_id": {"first": RUN_IDS.get("dash_10s"), "restart": RUN_IDS.get("dash_30s")},
    "run_id_unchanged_after_restart": q_dash.runId == q_dash_30.runId,
    "progress_10s": prog_10,
    "progress_30s": prog_30,
    "batches": BATCH_LOG,
    "sink_rows": n_sink,
    "duplicate_window_keys": n_dupes,
    "compared_windows": cmp["compared"],
    "matching_windows": cmp["matching"],
    "mismatching_windows": cmp["mismatching"],

    "replay_duplicates_without_checkpoint": both,
    "offsets": checkpoint_batches(CKPT / "dash"),
}
RESULT.write_text(json.dumps(out, indent=1, default=str))
print("written:", RESULT)
print()
print("numbers to cite in the report:")
print(f"  query id stable             : {out['query_id_stable']}")
print(f"  windows in sink             : {out['sink_rows']}")
print(f"  duplicated windows          : {out['duplicate_window_keys']}")
print(f"  stream == batch recompute   : {out['matching_windows']}/{out['compared_windows']}"
      f" matching, {out['mismatching_windows']} differing")
print(f"  windows replayed by a fresh : {out['replay_duplicates_without_checkpoint']}")

written: /tmp/spark-events/q4/q4_results.json

numbers to cite in the report:
  query id stable             : True
  windows in sink             : 107
  duplicated windows          : 0
  stream == batch recompute   : 63/63 matching, 0 differing
  windows replayed by a fresh : 0
